# $\nu_e$ CC Systematics Notebook v2

**Goal:** Compute covariance matrices for all three kinematic variables  
(`electron_ke`, `electron_p`, `electron_costheta`) and produce the standard  
systematic-uncertainty plots (universe spread, fractional-uncertainty budget,  
covariance/correlation heatmaps).

**Memory strategy:**
- Loads `sel_qual` (compact precomputed selection df) — avoids reloading the full `evtdf`
- External BNB/GENIE weights are aligned once and cached as a compressed `.npz`
- Covariances are accumulated *incrementally* — universe arrays are never kept in full
- Variables are processed one at a time; large intermediates are deleted explicitly

**Outputs saved to `nuecc_cov/`:**
- `{var}_flux_cov_matrices.npz`
- `{var}_genie_cov_matrices.npz`
- `{var}_mcstat_cov_matrices.npz`

each containing: `sig_cv`, `bkg_cv`, `true_cv`, `response`,  
`cov_ms_ms`, `cov_bs_bs`, `cov_ms_bs`, `frac_cov_ms`, `corr_ms`

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

## 0. Imports

In [ ]:
import os, sys, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm, Normalize
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

## 0.1 Paths & settings

In [ ]:
# ── cafpyana root ─────────────────────────────────────────────────────────────
CAFPYANA_WD = "/home/user/cafpyana"   # adjust if needed
NUE_DIR     = os.path.join(CAFPYANA_WD, "analysis_village/nueCC")

for p in [CAFPYANA_WD, CAFPYANA_WD + "/pyanalib", NUE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import nue_helpers   as nh
import nue_selection as ns

# ── Input files ───────────────────────────────────────────────────────────────
DF_FILE       = "/exp/sbnd/data/users/castalyf/nue_sel/production_files/gen1/mc1e20_nueCC_2605.df"
WEIGHTS_FILE  = "/exp/sbnd/data/users/castalyf/nue_sel/production_files/gen1/mc1e20_nueCC_weights_2605.df"
SEL_QUAL_FILE = os.path.join(NUE_DIR, "nuecc_dfs/selected_nuecc_qual_v2.df")

# ── Output directories ────────────────────────────────────────────────────────
COV_DIR    = os.path.join(NUE_DIR, "nuecc_cov")
PLOT_DIR   = os.path.join(NUE_DIR, "nuecc_syst_plots")
CACHE_DIR  = os.path.join(NUE_DIR, "nuecc_dfs")
WEIGHT_CACHE = os.path.join(CACHE_DIR, "aligned_weights_cache.npz")

for d in [COV_DIR, PLOT_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Settings ─────────────────────────────────────────────────────────────────
TARGET_POT     = 6.6e20
N_UNIVERSES    = 100    # universes per source
TITLE_PREFIX   = r"SBND $\nu_e$CC Inclusive"
SAVE_FIGS      = True

# Truth categories
SIG_CAT       = 0
EXCLUDE_CATS  = {7}                    # catch-all (not shown in histos)
BKG_CATS      = [1, 2, 3, 4, 5, 6]
STACK_ORDER   = [0, 1, 2, 3, 4, 5, 6]

# Final shower-quality cut column (last entry in ns.CUT_NAMES_MORE)
FINAL_SEL_COL = "sel_" + ns.CUT_NAMES_MORE[-1]   # e.g. 'sel_vertex_distance'

print(f"SEL_QUAL_FILE : {SEL_QUAL_FILE}")
print(f"WEIGHTS_FILE  : {WEIGHTS_FILE}")
print(f"COV_DIR       : {COV_DIR}")
print(f"FINAL_SEL_COL : {FINAL_SEL_COL}")

## 1. Load sel_qual (lightweight — no evtdf needed)

In [ ]:
print("Loading sel_qual ...")
sel_qual = pd.read_hdf(SEL_QUAL_FILE)
print(f"  shape  : {sel_qual.shape}")
print(f"  columns: {sel_qual.columns.tolist()}")
print(f"  index  : {sel_qual.index.names}")

# ── Final selection ───────────────────────────────────────────────────────────
sel_mask = sel_qual[FINAL_SEL_COL].fillna(False).astype(bool)
sel_df   = sel_qual[sel_mask].copy()

# Remove catch-all category 7 from analysis
n_cat7 = (sel_df["truth_cat"] == 7).sum()
sel_df = sel_df[~sel_df["truth_cat"].isin(EXCLUDE_CATS)].copy()

sig_mask  = sel_df["truth_cat"] == SIG_CAT    # selected signal
bkg_mask  = ~sig_mask                          # selected background
pot_scale = float(sel_qual["pot_scale"].iloc[0])

# Pre-selection signal: efficiency denominator (FM+FV, truth_cat==0, all cuts)
presel_sig = sel_qual[sel_qual["truth_cat"] == SIG_CAT]

print(f"\nSelected (raw)   : {sel_mask.sum():,}")
print(f"Cat-7 removed    : {n_cat7:,}")
print(f"Selected (final) : {len(sel_df):,}  "
      f"(sig={sig_mask.sum():,}, bkg={bkg_mask.sum():,})")
print(f"Purity           : {sig_mask.sum()/len(sel_df)*100:.1f}%")
print(f"pot_scale        : {pot_scale:.4f}")

gc.collect()

## 2. Variable configuration & binnings

In [ ]:
# ── Equal-statistics KE bins from signal data ─────────────────────────────────
def make_equal_stats_bins(vals, n_bins, lo, hi):
    """Build equal-statistics bins from a 1D array."""
    v = vals[(vals >= lo) & (vals < hi) & np.isfinite(vals)]
    edges = np.percentile(v, np.linspace(0, 100, n_bins + 1))
    edges[0], edges[-1] = lo, hi
    return np.unique(np.round(edges, 1))

_ke_sig = sel_df.loc[sig_mask, "reco_ke"].dropna().values
BINS_KE  = make_equal_stats_bins(_ke_sig, n_bins=6, lo=75., hi=1500.)

BINS_P   = np.array([100., 300., 500., 700., 900., 1200., 1500.])   # MeV/c
BINS_COS = np.linspace(-1., 1., 9)                                   # 8 bins

VAR_CONFIGS = [
    dict(name="electron_ke",
         reco_col="reco_ke",       true_col="true_ke",
         bins=BINS_KE,
         label=r"Leading $e^-$ KE [MeV]",
         label_reco=r"Reco leading $e^-$ KE [MeV]",
         label_true=r"True leading $e^-$ KE [MeV]"),
    dict(name="electron_p",
         reco_col="reco_p",        true_col="true_p",
         bins=BINS_P,
         label=r"Leading $e^-$ $p_e$ [MeV/c]",
         label_reco=r"Reco leading $e^-$ $p_e$ [MeV/c]",
         label_true=r"True leading $e^-$ $p_e$ [MeV/c]"),
    dict(name="electron_costheta",
         reco_col="reco_costheta", true_col="true_costheta",
         bins=BINS_COS,
         label=r"Leading $e^-$ $\cos\theta$",
         label_reco=r"Reco leading $e^-$ $\cos\theta$",
         label_true=r"True leading $e^-$ $\cos\theta$"),
]

print(f"KE bins [MeV]  : {np.round(BINS_KE, 1)}")
print(f"p  bins [MeV/c]: {BINS_P}")
print(f"cos bins       : {np.round(BINS_COS, 2)}")

## 3. CV histograms & response matrices (all variables)

In [ ]:
hists = {}   # keyed by var name

for vc in VAR_CONFIGS:
    rc, tc = vc["reco_col"], vc["true_col"]
    bins   = vc["bins"]
    lo, hi = bins[0], bins[-1]

    # Default-arg capture avoids late-binding closure bug in the loop
    def _in_range(vals, _lo=lo, _hi=hi):
        v = np.asarray(vals, dtype=float)
        return v[np.isfinite(v) & (v >= _lo) & (v < _hi)]

    def _hist(vals, w=pot_scale, _bins=bins):
        return np.histogram(vals, bins=_bins,
                            weights=np.full(len(vals), w))[0].astype(float)

    sig_reco = _in_range(sel_df.loc[sig_mask, rc].values)
    bkg_reco = _in_range(sel_df.loc[bkg_mask, rc].values)
    sig_true = _in_range(sel_df.loc[sig_mask, tc].values)
    pre_true = _in_range(presel_sig[tc].values)   # denominator (all presel signal)

    sig_cv      = _hist(sig_reco)             # POT-scaled selected signal
    bkg_cv      = _hist(bkg_reco)             # POT-scaled selected background
    true_cv     = _hist(pre_true, w=1.)       # unweighted true signal (eff denom)
    true_sel_cv = _hist(sig_true, w=1.)       # unweighted true signal (selected)

    # ── Response matrix R[true_bin, reco_bin] — column-normalised ────────────
    valid = sel_df[sig_mask].copy()
    vm    = valid[rc].notna() & valid[tc].notna()
    vm   &= (valid[rc] >= lo) & (valid[rc] < hi)
    vm   &= (valid[tc] >= lo) & (valid[tc] < hi)
    reco_vs_true, _, _ = np.histogram2d(
        valid.loc[vm, tc].values,
        valid.loc[vm, rc].values,
        bins=[bins, bins],
    )
    col_sums = reco_vs_true.sum(axis=1, keepdims=True).clip(1)
    response = (reco_vs_true / col_sums).astype(float)

    hists[vc["name"]] = dict(
        sig_cv=sig_cv, bkg_cv=bkg_cv,
        true_cv=true_cv, true_sel_cv=true_sel_cv,
        response=response, bins=bins,
    )
    print(f"{vc['name']:20s}  sig={sig_cv.sum():.1f}  bkg={bkg_cv.sum():.1f}  "
          f"true_denom={true_cv.sum():.0f}")

gc.collect()
print("\nCV histograms done.")

## 4. Load or align BNB / GENIE weights
First run: loads `evtdf` to extract `mct_index`, aligns to `sel_df`, saves `.npz` cache.  
Subsequent runs: loads cache directly — fast and memory-efficient.

In [ ]:
def _load_all_splits(hdf_file, key_prefix):
    """Load all split keys from an HDF5 file into a single DataFrame."""
    with pd.HDFStore(hdf_file, mode="r") as store:
        keys      = store.keys()
        has_split = "/split" in keys
        if has_split:
            n = int(store["split"]["n_split"].iloc[0])
            return pd.concat([store[f"{key_prefix}_{k}"] for k in range(n)], axis=0)
        matching = sorted([k for k in keys
                           if k.lstrip("/").startswith(key_prefix.lstrip("/"))])
        if not matching:
            return pd.DataFrame()
        return pd.concat([store[k] for k in matching], axis=0)


# ── Try cache first ────────────────────────────────────────────────────────────
if os.path.exists(WEIGHT_CACHE):
    print(f"Loading cached weights: {WEIGHT_CACHE}")
    cache         = np.load(WEIGHT_CACHE, allow_pickle=True)
    bnb_weights   = cache["bnb"].astype(np.float32)    # (n_sel, n_bnb)
    genie_weights = cache["genie"].astype(np.float32)  # (n_sel, n_genie)
    assert bnb_weights.shape[0] == len(sel_df), (
        f"Cache length mismatch: {bnb_weights.shape[0]} vs {len(sel_df)} — "
        f"delete {WEIGHT_CACHE} and re-run"
    )
    print(f"  BNB   : {bnb_weights.shape}")
    print(f"  GENIE : {genie_weights.shape}")

elif not os.path.exists(WEIGHTS_FILE):
    print(f"WARNING: WEIGHTS_FILE not found.  Proceeding with MCstat only.")
    bnb_weights   = np.ones((len(sel_df), 0), dtype=np.float32)
    genie_weights = np.ones((len(sel_df), 0), dtype=np.float32)

else:
    # ── Align from scratch ────────────────────────────────────────────────────
    print("Aligning weights (first-time setup) ...")

    # 1. Load mcnu weights
    print("  Loading mcnu_df ...")
    mcnu_df   = _load_all_splits(WEIGHTS_FILE, "mcnu")
    print(f"  mcnu_df: {mcnu_df.shape}")

    bnb_cols   = [c for c in mcnu_df.columns
                  if isinstance(c, tuple)
                  and any(kw in str(p).lower() for p in c
                          for kw in ["bnb", "flux", "beam"])
                  and any("univ" in str(p).lower() for p in c)]
    genie_cols = [c for c in mcnu_df.columns
                  if isinstance(c, tuple)
                  and any(kw in str(p).lower() for p in c
                          for kw in ["genie", "xsr", "knob"])
                  and any("univ" in str(p).lower() for p in c)]
    print(f"  BNB cols: {len(bnb_cols)},  GENIE cols: {len(genie_cols)}")

    # 2. Load evtdf minimally (only need mct_index per interaction)
    print("  Loading evtdf (interaction-level only) ...")
    evtdf = _load_all_splits(DF_FILE, "evt")
    il    = list(range(evtdf.index.nlevels - 1))
    inter_df = evtdf.groupby(level=il).first()
    del evtdf
    gc.collect()

    # 3. Find mct_index column
    mct_col = next(
        (c for c in inter_df.columns
         if "mct_index" in str(c) and "dlp_true" in str(c)), None
    ) or next(
        (c for c in inter_df.columns if "mct_index" in str(c)), None
    )
    print(f"  mct_index column: {mct_col}")

    if mct_col is None or (not bnb_cols and not genie_cols):
        print("  mct_index not found or no universe cols — MCstat only.")
        bnb_weights   = np.ones((len(sel_df), 0), dtype=np.float32)
        genie_weights = np.ones((len(sel_df), 0), dtype=np.float32)
    else:
        # 4. Align mct_index to sel_df interactions
        common = sel_df.index.intersection(inter_df.index)
        mct_series = inter_df.loc[common, mct_col].reindex(sel_df.index)
        del inter_df
        gc.collect()

        # 5. Merge mcnu weights via mct_index
        mcnu_names    = mcnu_df.index.names   # (__ntuple, entry, nu_idx)
        all_wgt_cols  = bnb_cols + genie_cols
        mcnu_slim     = mcnu_df[all_wgt_cols].reset_index()
        del mcnu_df
        gc.collect()

        # Flatten tuple column names to strings so they survive pd.merge cleanly
        def _col_str(c):
            return "__".join(str(x) for x in c) if isinstance(c, tuple) else str(c)

        bnb_flat   = [_col_str(c) for c in bnb_cols]
        genie_flat = [_col_str(c) for c in genie_cols]
        rn_wgt     = {c: _col_str(c) for c in all_wgt_cols}

        # Also rename the index columns we need
        rn_idx = {mcnu_names[0]: sel_df.index.names[0],
                  mcnu_names[1]: sel_df.index.names[1],
                  mcnu_names[2]: "mct_idx"}
        mcnu_slim = mcnu_slim.rename(columns={**rn_idx, **rn_wgt})

        # Build alignment df: sel_df index + mct_index for each selected interaction
        align_df = mct_series.reset_index()
        align_df.columns = list(sel_df.index.names) + ["mct_idx"]

        merged = align_df.merge(
            mcnu_slim,
            on=[sel_df.index.names[0], sel_df.index.names[1], "mct_idx"],
            how="left",
        )

        def _extract(flat_cols):
            if not flat_cols:
                return np.ones((len(sel_df), 0), dtype=np.float32)
            arr = merged[flat_cols].values.astype(np.float32)
            arr = np.clip(arr, 0., 10.)
            arr[~np.isfinite(arr)] = 1.0
            return arr

        bnb_weights   = _extract(bnb_flat)
        genie_weights = _extract(genie_flat)

        del mcnu_slim, merged, align_df, mct_series
        gc.collect()

        # 6. Save cache
        np.savez_compressed(
            WEIGHT_CACHE,
            bnb=bnb_weights,
            genie=genie_weights,
        )
        print(f"  Cache saved → {WEIGHT_CACHE}")

N_BNB   = bnb_weights.shape[1]
N_GENIE = genie_weights.shape[1]
print(f"\nBNB universes  : {N_BNB}")
print(f"GENIE universes: {N_GENIE}")

## 5. Build universe histograms & covariance matrices
Covariances are accumulated *incrementally* per universe — only O(n_bins²) memory.

In [ ]:
# ── Helper: incremental covariance accumulation ───────────────────────────────
def compute_cov_incremental(sel_df, sig_mask, bkg_mask, bins,
                             reco_col, weight_arr, cv_sig, cv_bkg,
                             pot_scale, n_univ, desc="", store_univs=True):
    """
    Compute covariance matrices from universe weights without storing all
    universe histograms simultaneously.

    Returns
    -------
    dict with Cov_ss, Cov_bb, Cov_sb, FracCov_ss, Corr_ss,
    and (if store_univs) sig_univs (n_univ, n_bins) and bkg_univs.
    """
    lo, hi  = bins[0], bins[-1]
    n_bins  = len(bins) - 1
    n_sel   = len(sel_df)

    # Range masks for reco values
    rc_vals  = sel_df[reco_col].values.astype(float)
    rng_mask = np.isfinite(rc_vals) & (rc_vals >= lo) & (rc_vals < hi)

    sig_idx  = np.where(sig_mask.values & rng_mask)[0]
    bkg_idx  = np.where(bkg_mask.values & rng_mask)[0]

    sig_vals = rc_vals[sig_idx]
    bkg_vals = rc_vals[bkg_idx]

    Cov_ss   = np.zeros((n_bins, n_bins))
    Cov_bb   = np.zeros((n_bins, n_bins))
    Cov_sb   = np.zeros((n_bins, n_bins))

    if store_univs:
        sig_univs = np.zeros((n_univ, n_bins), dtype=np.float32)
        bkg_univs = np.zeros((n_univ, n_bins), dtype=np.float32)

    for u in tqdm(range(n_univ), desc=desc, leave=False):
        if weight_arr.shape[1] == 0:
            w_sig = np.ones(len(sig_idx), dtype=float) * pot_scale
            w_bkg = np.ones(len(bkg_idx), dtype=float) * pot_scale
        else:
            u_idx = u % weight_arr.shape[1]   # wrap if fewer than n_univ
            w_sig = weight_arr[sig_idx, u_idx].astype(float) * pot_scale
            w_bkg = weight_arr[bkg_idx, u_idx].astype(float) * pot_scale

        sig_u, _ = np.histogram(sig_vals, bins=bins, weights=w_sig)
        bkg_u, _ = np.histogram(bkg_vals, bins=bins, weights=w_bkg)

        d_sig = sig_u.astype(float) - cv_sig
        d_bkg = bkg_u.astype(float) - cv_bkg
        Cov_ss += np.outer(d_sig, d_sig)
        Cov_bb += np.outer(d_bkg, d_bkg)
        Cov_sb += np.outer(d_sig, d_bkg)

        if store_univs:
            sig_univs[u] = sig_u.astype(np.float32)
            bkg_univs[u] = bkg_u.astype(np.float32)

    Cov_ss  /= n_univ
    Cov_bb  /= n_univ
    Cov_sb  /= n_univ

    diag_sig = np.sqrt(np.diag(Cov_ss))
    with np.errstate(divide="ignore", invalid="ignore"):
        FracCov = Cov_ss / np.outer(
            np.where(cv_sig > 0, cv_sig, np.nan),
            np.where(cv_sig > 0, cv_sig, np.nan),
        )
        denom   = np.where(diag_sig > 0, diag_sig, np.nan)
        Corr    = Cov_ss / np.outer(denom, denom)

    out = dict(Cov_ss=Cov_ss, Cov_bb=Cov_bb, Cov_sb=Cov_sb,
               FracCov_ss=FracCov, Corr_ss=Corr)
    if store_univs:
        out["sig_univs"] = sig_univs
        out["bkg_univs"] = bkg_univs
    return out


# ── MCstat: Poisson resampling (no external weights needed) ───────────────────
def compute_mcstat_univs(sel_df, sig_mask, bkg_mask, bins, reco_col,
                          cv_sig, cv_bkg, pot_scale, n_univ=100, seed=42,
                          store_univs=True):
    lo, hi  = bins[0], bins[-1]
    n_bins  = len(bins) - 1
    n_sel   = len(sel_df)

    rc_vals  = sel_df[reco_col].values.astype(float)
    rng_mask = np.isfinite(rc_vals) & (rc_vals >= lo) & (rc_vals < hi)
    sig_idx  = np.where(sig_mask.values & rng_mask)[0]
    bkg_idx  = np.where(bkg_mask.values & rng_mask)[0]
    sig_vals = rc_vals[sig_idx]
    bkg_vals = rc_vals[bkg_idx]

    rng = np.random.default_rng(seed)
    seeds_u = rng.integers(0, 2**31, size=n_univ, dtype=np.uint32)

    Cov_ss  = np.zeros((n_bins, n_bins))
    Cov_bb  = np.zeros((n_bins, n_bins))
    Cov_sb  = np.zeros((n_bins, n_bins))

    if store_univs:
        sig_univs = np.zeros((n_univ, n_bins), dtype=np.float32)
        bkg_univs = np.zeros((n_univ, n_bins), dtype=np.float32)

    for u in tqdm(range(n_univ), desc="MCstat", leave=False):
        rng_u   = np.random.default_rng(int(seeds_u[u]))
        # Draw Poisson weights for ALL selected events
        w_all   = rng_u.poisson(1.0, size=n_sel).astype(float) * pot_scale

        sig_u, _ = np.histogram(sig_vals, bins=bins, weights=w_all[sig_idx])
        bkg_u, _ = np.histogram(bkg_vals, bins=bins, weights=w_all[bkg_idx])

        d_sig = sig_u.astype(float) - cv_sig
        d_bkg = bkg_u.astype(float) - cv_bkg
        Cov_ss += np.outer(d_sig, d_sig)
        Cov_bb += np.outer(d_bkg, d_bkg)
        Cov_sb += np.outer(d_sig, d_bkg)

        if store_univs:
            sig_univs[u] = sig_u.astype(np.float32)
            bkg_univs[u] = bkg_u.astype(np.float32)

    Cov_ss /= n_univ
    Cov_bb /= n_univ
    Cov_sb /= n_univ

    diag_sig = np.sqrt(np.diag(Cov_ss))
    with np.errstate(divide="ignore", invalid="ignore"):
        FracCov = Cov_ss / np.outer(
            np.where(cv_sig > 0, cv_sig, np.nan),
            np.where(cv_sig > 0, cv_sig, np.nan),
        )
        denom   = np.where(diag_sig > 0, diag_sig, np.nan)
        Corr    = Cov_ss / np.outer(denom, denom)

    out = dict(Cov_ss=Cov_ss, Cov_bb=Cov_bb, Cov_sb=Cov_sb,
               FracCov_ss=FracCov, Corr_ss=Corr)
    if store_univs:
        out["sig_univs"] = sig_univs
        out["bkg_univs"] = bkg_univs
    return out

print("Universe + covariance helpers defined.")

In [ ]:
# ── Run for all variables × all sources ──────────────────────────────────────
# Results stored as: covs[var_name][source] = dict(...)
covs = {vc["name"]: {} for vc in VAR_CONFIGS}

for vc in VAR_CONFIGS:
    vname  = vc["name"]
    rc     = vc["reco_col"]
    bins   = vc["bins"]
    h      = hists[vname]
    sig_cv = h["sig_cv"]
    bkg_cv = h["bkg_cv"]

    print(f"\n{'='*55}")
    print(f"  Variable: {vname}")
    print(f"{'='*55}")

    # BNB Flux
    if N_BNB > 0:
        print(f"  BNB flux ({N_BNB} univs) ...")
        covs[vname]["flux"] = compute_cov_incremental(
            sel_df, sig_mask, bkg_mask, bins, rc,
            bnb_weights, sig_cv, bkg_cv,
            pot_scale, N_BNB, desc=f"BNB [{vname}]"
        )
        frac_unc = np.sqrt(np.diag(covs[vname]["flux"]["Cov_ss"]))
        print(f"  Flux frac. unc: {np.round(frac_unc / np.where(sig_cv>0,sig_cv,np.nan), 3)}")
    else:
        print("  BNB: skipped (no universe weights)")
        covs[vname]["flux"] = None

    # GENIE
    if N_GENIE > 0:
        print(f"  GENIE ({N_GENIE} univs) ...")
        covs[vname]["genie"] = compute_cov_incremental(
            sel_df, sig_mask, bkg_mask, bins, rc,
            genie_weights, sig_cv, bkg_cv,
            pot_scale, N_GENIE, desc=f"GENIE [{vname}]"
        )
        frac_unc = np.sqrt(np.diag(covs[vname]["genie"]["Cov_ss"]))
        print(f"  GENIE frac. unc: {np.round(frac_unc / np.where(sig_cv>0,sig_cv,np.nan), 3)}")
    else:
        print("  GENIE: skipped (no universe weights)")
        covs[vname]["genie"] = None

    # MCstat
    print(f"  MCstat ({N_UNIVERSES} Poisson univs) ...")
    covs[vname]["mcstat"] = compute_mcstat_univs(
        sel_df, sig_mask, bkg_mask, bins, rc,
        sig_cv, bkg_cv, pot_scale, n_univ=N_UNIVERSES
    )
    frac_unc = np.sqrt(np.diag(covs[vname]["mcstat"]["Cov_ss"]))
    print(f"  MCstat frac. unc: {np.round(frac_unc / np.where(sig_cv>0,sig_cv,np.nan), 3)}")

    # Combined total covariance (sum of sources)
    total_cov = covs[vname]["mcstat"]["Cov_ss"].copy()
    if covs[vname]["flux"]  is not None: total_cov += covs[vname]["flux"]["Cov_ss"]
    if covs[vname]["genie"] is not None: total_cov += covs[vname]["genie"]["Cov_ss"]
    total_frac = np.sqrt(np.diag(total_cov)) / np.where(sig_cv > 0, sig_cv, np.nan)
    print(f"  Total frac. unc:  {np.round(total_frac, 3)}")
    covs[vname]["total_cov"] = total_cov

print("\nAll covariances computed.")

## 6. Save covariance matrices to `.npz`

In [ ]:
def save_cov_npz(path, sig_cv, bkg_cv, true_cv, response, cov_dict):
    """
    Save covariance matrices to .npz in the format expected by
    nueCC_xsec_analysis.ipynb.

    Each covariance entry is stored as a numpy object array containing a dict
    {"cov": matrix} so that it can be retrieved with npz[key].item()["cov"].
    """
    def _wrap(mat):
        obj = np.empty(1, dtype=object)
        obj[0] = {"cov": mat}
        return obj

    save_dict = dict(
        ms          = sig_cv,
        bs          = bkg_cv,
        true_signal = true_cv,
        response    = response,
        cov_ms_ms   = _wrap(cov_dict["Cov_ss"]),
        cov_bs_bs   = _wrap(cov_dict["Cov_bb"]),
        cov_ms_bs   = _wrap(cov_dict["Cov_sb"]),
        frac_cov_ms = cov_dict["FracCov_ss"],
        corr_ms     = cov_dict["Corr_ss"],
    )
    np.savez(path, **save_dict)
    kb = os.path.getsize(path + ".npz") / 1024 if os.path.exists(path + ".npz") \
        else os.path.getsize(path) / 1024
    print(f"  Saved → {path}  ({kb:.0f} KB)")


source_map = [
    ("flux",   "flux_cov_matrices"),
    ("genie",  "genie_cov_matrices"),
    ("mcstat", "mcstat_cov_matrices"),
]

for vc in VAR_CONFIGS:
    vname = vc["name"]
    h     = hists[vname]
    print(f"\n{vname}:")
    for src, fname_suffix in source_map:
        cov = covs[vname].get(src)
        if cov is None:
            print(f"  {src}: skipped")
            continue
        fpath = os.path.join(COV_DIR, f"{vname}_{fname_suffix}")
        save_cov_npz(fpath, h["sig_cv"], h["bkg_cv"],
                     h["true_cv"], h["response"], cov)

# Legacy compatibility: alias electron_ke files to the old names
for src, fname_suffix in source_map:
    src_path  = os.path.join(COV_DIR, f"electron_ke_{fname_suffix}.npz")
    dest_path = os.path.join(COV_DIR, f"{fname_suffix}.npz")
    if os.path.exists(src_path) and not os.path.exists(dest_path):
        import shutil
        shutil.copy2(src_path, dest_path)
        print(f"Legacy alias: {fname_suffix}.npz → electron_ke")

print("\nAll .npz files saved.")

## 7. Plots

### 7.1 Stacked topology histograms (selected)

In [ ]:
def savefig(fname):
    if SAVE_FIGS:
        plt.savefig(os.path.join(PLOT_DIR, fname), bbox_inches="tight", dpi=150)

def _bin_width_weights(vals, bins, scalar_weight=1.0):
    vals   = np.asarray(vals, dtype=float)
    widths = np.diff(bins)
    idx    = np.clip(np.digitize(vals, bins) - 1, 0, len(widths) - 1)
    return scalar_weight / widths[idx]

_YLABELS = {
    "electron_ke":       "Events / MeV",
    "electron_p":        "Events / (MeV/c)",
    "electron_costheta": r"Events / $\Delta\!\cos\theta$",
}

for vc in VAR_CONFIGS:
    rc, bins = vc["reco_col"], vc["bins"]
    lo, hi   = bins[0], bins[-1]

    def _cat_vals(cat):
        m = (sel_df["truth_cat"] == cat) & sel_df[rc].between(lo, hi - 1e-8)
        return sel_df.loc[m, rc].values

    series_list  = [_cat_vals(c) for c in STACK_ORDER]
    weights_list = [_bin_width_weights(s, bins, pot_scale) for s in series_list]

    fig, ax = plt.subplots(figsize=(8, 6))
    nh.plot_stacked_hist(
        series_list        = series_list,
        labels             = [nh.CAT_LABELS[c] for c in STACK_ORDER],
        colors             = [nh.CAT_COLORS[c] for c in STACK_ORDER],
        bins               = bins,
        weights            = weights_list,
        xlabel             = vc["label_reco"],
        ylabel             = _YLABELS.get(vc["name"], "Events / bin"),
        title              = fr"{TITLE_PREFIX}: After shower-quality cuts",
        pot_label          = fr"${TARGET_POT:.1e}$ POT",
        ax                 = ax,
        invert_stack_order = False,
    )
    # Overlay total uncertainty band on signal+background
    total_cov = covs[vc["name"]]["total_cov"]
    h_tot, _  = np.histogram(
        np.clip(sel_df.loc[sel_df[rc].notna(), rc], lo, hi - 1e-8),
        bins=bins, weights=_bin_width_weights(
            np.clip(sel_df.loc[sel_df[rc].notna(), rc], lo, hi - 1e-8),
            bins, pot_scale)
    )
    bin_cents = 0.5 * (bins[:-1] + bins[1:])
    bin_widths = np.diff(bins)
    # Scale cov to density
    density_cov = total_cov / np.outer(bin_widths, bin_widths)
    unc_density  = np.sqrt(np.diag(density_cov))
    ax.fill_between(
        np.repeat(bins, 2)[1:-1],
        np.repeat(h_tot - unc_density, 2),
        np.repeat(h_tot + unc_density, 2),
        alpha=0.25, color="gray", label="Total unc."
    )
    ax.legend(fontsize=8, ncol=2, loc="upper right")
    plt.tight_layout()
    savefig(f"{vc['name']}_stack_sel.png")
    plt.show()

### 7.2 Universe spread plots (like `plot_all_distributions`)

In [ ]:
SOURCE_COLORS = {"flux": "C0", "genie": "C1", "mcstat": "C2"}
SOURCE_LABELS = {"flux": "BNB Flux", "genie": "GENIE", "mcstat": "MCstat"}

for vc in VAR_CONFIGS:
    vname = vc["name"]
    bins  = vc["bins"]
    h     = hists[vname]
    sig_cv = h["sig_cv"]

    sources_with_univs = [
        src for src in ["flux", "genie", "mcstat"]
        if covs[vname].get(src) is not None
        and "sig_univs" in covs[vname][src]
    ]

    n_src = len(sources_with_univs)
    if n_src == 0:
        continue

    fig, axes = plt.subplots(1, n_src, figsize=(6 * n_src, 5), sharey=False)
    if n_src == 1:
        axes = [axes]

    for ax, src in zip(axes, sources_with_univs):
        sig_univs = covs[vname][src]["sig_univs"]   # (n_univ, n_bins)
        color     = SOURCE_COLORS[src]
        label     = SOURCE_LABELS[src]

        # Draw universe spread (first 60 or all)
        n_show = min(60, sig_univs.shape[0])
        for u in range(n_show):
            ax.step(bins, np.append(sig_univs[u], sig_univs[u, -1]),
                    where="post", color=color, alpha=0.15, lw=0.6)

        # 1-sigma band
        mean_u = sig_univs.mean(axis=0)
        std_u  = sig_univs.std(axis=0)
        ax.fill_between(
            np.repeat(bins, 2)[1:-1],
            np.repeat(mean_u - std_u, 2),
            np.repeat(mean_u + std_u, 2),
            alpha=0.35, color=color, label=r"$\pm1\sigma$ spread"
        )
        ax.step(bins, np.append(sig_cv, sig_cv[-1]),
                where="post", color="black", lw=2, label="CV")

        ax.set_xlabel(vc["label_reco"], fontsize=11)
        ax.set_ylabel("Events (POT-scaled)", fontsize=11)
        ax.set_title(f"{TITLE_PREFIX}\n{label} universes — {vname}", fontsize=10)
        ax.legend(fontsize=9, frameon=False)
        ax.grid(alpha=0.2, ls="--")
        ax.set_xlim(bins[0], bins[-1])

    plt.tight_layout()
    savefig(f"{vname}_universe_spread.png")
    plt.show()

### 7.3 Fractional uncertainty budget (like `plot_event_rate_errs`)

In [ ]:
for vc in VAR_CONFIGS:
    vname  = vc["name"]
    bins   = vc["bins"]
    h      = hists[vname]
    sig_cv = h["sig_cv"]
    bkg_cv = h["bkg_cv"]
    total  = sig_cv + bkg_cv

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, cv, label_cv in zip(axes, [sig_cv, total],
                                 ["Signal", "Signal+Background"]):
        denom = np.where(cv > 0, cv, np.nan)

        # Plot individual sources
        for src in ["flux", "genie", "mcstat"]:
            cov_data = covs[vname].get(src)
            if cov_data is None:
                continue
            frac = np.sqrt(np.diag(cov_data["Cov_ss"])) / denom * 100.
            ax.step(bins, np.append(frac, frac[-1]),
                    where="post",
                    label=SOURCE_LABELS[src],
                    color=SOURCE_COLORS[src], lw=1.5)

        # Total
        total_frac = np.sqrt(np.diag(covs[vname]["total_cov"])) / denom * 100.
        ax.step(bins, np.append(total_frac, total_frac[-1]),
                where="post", label="Total", color="black", lw=2.5, ls="--")

        ax.set_xlabel(vc["label_reco"], fontsize=12)
        ax.set_ylabel("Fractional uncertainty [%]", fontsize=12)
        ax.set_title(f"{TITLE_PREFIX} — {label_cv} uncertainty budget\n{vname}",
                     fontsize=10)
        ax.set_xlim(bins[0], bins[-1])
        ax.set_ylim(0, None)
        ax.legend(frameon=False, fontsize=10)
        ax.grid(alpha=0.2, ls="--")

    plt.tight_layout()
    savefig(f"{vname}_fracunc_budget.png")
    plt.show()

    # ── Summary table ──────────────────────────────────────────────────────────
    print(f"\n{'─'*60}")
    print(f"  {vname} — total signal fractional uncertainty per bin:")
    print(f"{'─'*60}")
    for i, (lo_b, hi_b) in enumerate(zip(bins[:-1], bins[1:])):
        s = f"  [{lo_b:.1f}, {hi_b:.1f}]  Total: {total_frac[i]:5.1f}%"
        for src in ["flux", "genie", "mcstat"]:
            cov_data = covs[vname].get(src)
            if cov_data is None:
                continue
            frac_i = np.sqrt(np.diag(cov_data["Cov_ss"]))[i] / (
                sig_cv[i] if sig_cv[i] > 0 else np.nan) * 100.
            s += f"  {SOURCE_LABELS[src]}: {frac_i:5.1f}%"
        print(s)

### 7.4 Covariance & correlation heatmaps (like `plot_all_covariance_matrices`)

In [ ]:
def _plot_matrix(mat, title, xlabel, bins, fmt=".3f", cmap="cividis",
                 vmin=None, vmax=None, symmetric_clim=False):
    n = mat.shape[0]
    fig, ax = plt.subplots(figsize=(7, 6))
    if symmetric_clim:
        m = np.nanmax(np.abs(mat))
        norm = TwoSlopeNorm(vmin=-m, vcenter=0, vmax=m)
        cmap = "RdBu_r"
    else:
        norm = Normalize(vmin=vmin, vmax=vmax)
    im = ax.pcolormesh(bins, bins, mat, norm=norm, cmap=cmap)
    plt.colorbar(im, ax=ax)

    # Annotate cells
    cents = 0.5 * (bins[:-1] + bins[1:])
    for i in range(n):
        for j in range(n):
            v = mat[i, j]
            if np.isfinite(v):
                txt_color = "white" if abs(v) > 0.6 * np.nanmax(np.abs(mat)) else "black"
                ax.text(cents[j], cents[i], format(v, fmt),
                        ha="center", va="center", fontsize=7, color=txt_color)

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel(xlabel, fontsize=11)
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    return fig, ax


for vc in VAR_CONFIGS:
    vname  = vc["name"]
    bins   = vc["bins"]
    xlabel = vc["label_reco"]

    for src in ["flux", "genie", "mcstat"]:
        cov_data = covs[vname].get(src)
        if cov_data is None:
            continue
        label = SOURCE_LABELS[src]

        # Covariance (signal × signal)
        fig, _ = _plot_matrix(
            cov_data["Cov_ss"],
            f"{TITLE_PREFIX}\n{label} Covariance (sig×sig) — {vname}",
            xlabel, bins, fmt=".2e", cmap="cividis"
        )
        savefig(f"{vname}_{src}_covariance.png")
        plt.show()

        # Fractional covariance
        fig, _ = _plot_matrix(
            cov_data["FracCov_ss"],
            f"{TITLE_PREFIX}\n{label} Frac. Covariance (sig×sig) — {vname}",
            xlabel, bins, fmt=".3f", cmap="cividis"
        )
        savefig(f"{vname}_{src}_frac_covariance.png")
        plt.show()

        # Correlation
        fig, _ = _plot_matrix(
            cov_data["Corr_ss"],
            f"{TITLE_PREFIX}\n{label} Correlation (sig×sig) — {vname}",
            xlabel, bins, fmt=".2f", symmetric_clim=True
        )
        savefig(f"{vname}_{src}_correlation.png")
        plt.show()

    # Total covariance
    total_cov = covs[vname]["total_cov"]
    denom_sq  = np.outer(
        np.where(hists[vname]["sig_cv"] > 0, hists[vname]["sig_cv"], np.nan),
        np.where(hists[vname]["sig_cv"] > 0, hists[vname]["sig_cv"], np.nan),
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        total_fraccov = total_cov / denom_sq
    fig, _ = _plot_matrix(
        total_fraccov,
        f"{TITLE_PREFIX}\nTotal Frac. Covariance — {vname}",
        xlabel, bins, fmt=".3f", cmap="cividis"
    )
    savefig(f"{vname}_total_frac_covariance.png")
    plt.show()

### 7.5 Purity & efficiency per bin

In [ ]:
for vc in VAR_CONFIGS:
    vname = vc["name"]
    bins  = vc["bins"]
    h     = hists[vname]

    purity = np.divide(h["sig_cv"], h["sig_cv"] + h["bkg_cv"],
                       out=np.zeros(len(h["sig_cv"])),
                       where=(h["sig_cv"] + h["bkg_cv"]) > 0)
    eff    = np.divide(h["true_sel_cv"], h["true_cv"],
                       out=np.zeros(len(h["true_cv"])),
                       where=h["true_cv"] > 0)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, vals, ylabel, color, glbl in [
        (axes[0], purity * 100, "Purity [%]",      "steelblue",
         f"Global: {h['sig_cv'].sum() / max(h['sig_cv'].sum() + h['bkg_cv'].sum(), 1) * 100:.1f}%"),
        (axes[1], eff    * 100, "Efficiency [%]",  "crimson",
         f"Global: {h['true_sel_cv'].sum() / max(h['true_cv'].sum(), 1) * 100:.1f}%"),
    ]:
        ax.step(bins, np.append(vals, vals[-1]), where="post", color=color, lw=2)
        ax.fill_between(bins, np.append(vals, vals[-1]),
                        step="post", alpha=0.15, color=color)
        ax.set_xlim(bins[0], bins[-1])
        ax.set_ylim(0, 110)
        ax.set_xlabel(vc["label"], fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.grid(alpha=0.2, ls="--")
        ax.text(0.05, 0.95, glbl, transform=ax.transAxes,
                va="top", fontsize=11, bbox=dict(fc="white", alpha=0.6, ec="none"))

    axes[0].set_title(f"{TITLE_PREFIX}: Signal Purity — {vname}")
    axes[1].set_title(f"{TITLE_PREFIX}: Signal Efficiency — {vname}")
    plt.tight_layout()
    savefig(f"{vname}_purity_efficiency.png")
    plt.show()

### 7.6 Response matrix per variable

In [ ]:
for vc in VAR_CONFIGS:
    vname  = vc["name"]
    bins   = vc["bins"]
    h      = hists[vname]
    R      = h["response"]   # (n_true_bins, n_reco_bins)

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.pcolormesh(bins, bins, R, cmap="Blues", vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label="Normalised response")

    cents = 0.5 * (bins[:-1] + bins[1:])
    for i in range(len(cents)):
        for j in range(len(cents)):
            v = R[i, j]
            if v > 0.01:
                ax.text(cents[j], cents[i], f"{v:.2f}",
                        ha="center", va="center", fontsize=7,
                        color="white" if v > 0.5 else "black")

    ax.set_xlabel(vc["label_reco"], fontsize=12)
    ax.set_ylabel(vc["label_true"], fontsize=12)
    ax.set_title(f"{TITLE_PREFIX}: Response Matrix — {vname}", fontsize=11)
    plt.tight_layout()
    savefig(f"{vname}_response_matrix.png")
    plt.show()

    # Column sums (should be ≤ 1, sum of efficiency per true bin)
    print(f"{vname} — response column sums (efficiency per true bin):")
    print(f"  {np.round(R.sum(axis=0), 3)}")

### 7.7 All-variable combined uncertainty summary

In [ ]:
fig, axes = plt.subplots(1, len(VAR_CONFIGS), figsize=(6 * len(VAR_CONFIGS), 5))
if len(VAR_CONFIGS) == 1:
    axes = [axes]

for ax, vc in zip(axes, VAR_CONFIGS):
    vname  = vc["name"]
    bins   = vc["bins"]
    sig_cv = hists[vname]["sig_cv"]
    denom  = np.where(sig_cv > 0, sig_cv, np.nan)

    for src in ["flux", "genie", "mcstat"]:
        cov_data = covs[vname].get(src)
        if cov_data is None:
            continue
        frac = np.sqrt(np.diag(cov_data["Cov_ss"])) / denom * 100.
        ax.step(bins, np.append(frac, frac[-1]),
                where="post", label=SOURCE_LABELS[src],
                color=SOURCE_COLORS[src], lw=1.5)

    total_frac = np.sqrt(np.diag(covs[vname]["total_cov"])) / denom * 100.
    ax.step(bins, np.append(total_frac, total_frac[-1]),
            where="post", label="Total", color="black", lw=2.5, ls="--")

    ax.set_xlabel(vc["label_reco"], fontsize=11)
    ax.set_ylabel("Signal frac. unc. [%]", fontsize=11)
    ax.set_title(vname, fontsize=10)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, None)
    ax.legend(frameon=False, fontsize=9)
    ax.grid(alpha=0.2, ls="--")

fig.suptitle(fr"{TITLE_PREFIX} — Systematic uncertainty budget", fontsize=12, y=1.02)
plt.tight_layout()
savefig("all_vars_fracunc_budget.png")
plt.show()

print("\nAll plots done.")
print(f"Figures saved to: {PLOT_DIR}")
print(f"Covariances saved to: {COV_DIR}")